# KazFinPersona — running a real model on Colab

This notebook runs an LLM over the test split and writes `predictions.jsonl`
in exactly the format `scripts/evaluate.py` already reads.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

**Why checkpointing matters.** A free Colab session can be cut off at any
point. Every 25 items are flushed to Google Drive, and a restarted run skips
what is already done. Without this you lose an hour of GPU time to a dropped
connection.

**What to expect.** The interesting number is not the model's accuracy on its
own. It is the comparison against `profile_blind` — a stub that ignores the
client entirely and still scores 0.708 product-match. A model that does not
clearly beat that on *counterfactual correctness* is not personalising,
however fluent its answers are.

## 1. Setup

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install transformers accelerate bitsandbytes sentencepiece 2>&1 | tail -2

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

WORK = '/content/drive/MyDrive/kazfinpersona'
import os
os.makedirs(WORK, exist_ok=True)
print('working dir:', WORK)

## 2. Get the code and data

The repo is private, so cloning needs a token. Simpler for now: build the
dataset here. It is deterministic given the seed, so it reproduces exactly
what you have locally.

**Option A** — upload the repo as a zip (left sidebar → Files → upload), then:

In [ ]:
# Option A: unzip an uploaded repo archive
# !unzip -q -o /content/kazfinpersona.zip -d /content/

# Option B: clone (public repo, or use a token URL for a private one)
# !git clone https://github.com/ShaikhiyevaZhazira/kazfinpersona.git /content/kazfinpersona

REPO = '/content/kazfinpersona'
assert os.path.isdir(REPO), 'upload or clone the repo first'
%cd $REPO
!pip -q install -r requirements.txt 2>&1 | tail -1

In [ ]:
# Dataset caching.
#
# /content is WIPED when a Colab session ends, so the dataset has to be either
# rebuilt or restored on every reconnect. An earlier version of this cell used
# `cp -r SRC DST` without creating the parent directory: the copy failed, but
# the script still printed 'restored from Drive' and the next cell died with
# FileNotFoundError. A cache that lies about succeeding is worse than no cache.
#
# This version checks for the actual FILES, not just the directory, and asserts
# at the end. It cannot report success without having succeeded.

import os

SEED_DATA  = 42
TARGET     = f'{REPO}/data/v0.5'
DATA_CACHE = f'{WORK}/data_v0.5_seed{SEED_DATA}'
REQUIRED   = ['clients.jsonl', 'clients_counterfactual.jsonl',
              'transactions.jsonl', 'scenarios.jsonl', 'segments.json']

def complete(d):
    return all(os.path.exists(f'{d}/{f}') for f in REQUIRED)

os.makedirs(f'{REPO}/data', exist_ok=True)

if complete(TARGET):
    print('dataset already present locally')
elif complete(DATA_CACHE):
    os.makedirs(TARGET, exist_ok=True)
    !cp -r "$DATA_CACHE/." "$TARGET/"
    print('restored from Drive')
else:
    print('building from scratch (2-3 min)')
    !python scripts/build_dataset.py --n-clients 2000 --seed $SEED_DATA --out data/v0.5 2>&1 | tail -4
    !python scripts/fit_segments.py  --data data/v0.5 --seed $SEED_DATA 2>&1 | tail -6

missing = [f for f in REQUIRED if not os.path.exists(f'{TARGET}/{f}')]
assert not missing, f'dataset incomplete, missing: {missing}'

# keep a copy in Drive for the next session
if not complete(DATA_CACHE):
    os.makedirs(DATA_CACHE, exist_ok=True)
    !cp -r "$TARGET/." "$DATA_CACHE/"
    print('cached to Drive')

print('OK:', sorted(os.listdir(TARGET)))


## 3. Choose the model

| Model | Size | Fits T4 (4-bit) | Note |
|---|---|---|---|
| `Qwen/Qwen2.5-3B-Instruct` | 3B | comfortably | start here |
| `Qwen/Qwen2.5-7B-Instruct` | 7B | yes | slower, better |
| `issai/LLama-3.1-KazLLM-1.0-8B` | 8B | tight | Kazakh-centric |
| `inceptionai/Llama-3.1-Sherkala-8B-Chat` | 8B | tight | Kazakh-centric

Run the 3B first to confirm the loop works end to end, then scale up.

**Include at least one Kazakh-centric model** or state in Limitations why you
could not — a reviewer of a Kazakh-language paper will ask why the experiments
use a general multilingual model.

In [ ]:
MODEL_ID  = 'Qwen/Qwen2.5-3B-Instruct'
CONTEXT   = 'profile'    # none | persona | profile | full
CATALOGUE = 'listed'     # closed | listed | filtered
LANGUAGE  = 'kk'         # kk | ru | mixed
SEED      = 0
LIMIT     = 200          # None = full test split (1104 items, ~3.7h on T4)

BASE_ID = (f"{MODEL_ID.split('/')[-1]}__ctx-{CONTEXT}__cat-{CATALOGUE}"
          f"__lang-{LANGUAGE}__seed{SEED}")
print('base id:', BASE_ID)
print('RUN_ID and checkpoint path are finalised in the sampling cell,')
print('because they must include a fingerprint of the evaluated item set.')


In [ ]:
import torch, json, sys, time, datetime as dt, collections
sys.path.insert(0, f'{REPO}/src')
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb = BitsAndBytesConfig(load_in_4bit=True,
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_quant_type='nf4',
                         bnb_4bit_use_double_quant=True)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map='auto', dtype=torch.float16)
model.eval()
torch.manual_seed(SEED)
print('loaded:', MODEL_ID)

## 4. Load scenarios and build prompts

In [ ]:
from kfp import eligibility, prompts, suitability
from kfp.eval.parsing import AnswerParser
from kfp.schemas import ClientProfile, Transaction

DATA = f'{REPO}/data/v0.5'
AS_OF = dt.date(2026, 6, 1)
catalog = eligibility.load_catalog(f'{REPO}/config/products_halyk.json')

clients = {}
for fn in ('clients.jsonl', 'clients_counterfactual.jsonl'):
    p = f'{DATA}/{fn}'
    if os.path.exists(p):
        for line in open(p, encoding='utf-8'):
            d = json.loads(line); clients[d['client_id']] = ClientProfile(**d)

tx = collections.defaultdict(list)
for line in open(f'{DATA}/transactions.jsonl', encoding='utf-8'):
    d = json.loads(line); tx[d['client_id']].append(Transaction(**d))

# counterfactuals inherit the origin's history (see fit_segments.py)
feats = {}
for cid, p in clients.items():
    own = tx.get(cid, []) or tx.get(cid.split('-CF-')[0], [])
    feats[cid] = suitability.behavioural_features(p, own, AS_OF)

segs = json.load(open(f'{DATA}/segments.json', encoding='utf-8')) \
       if os.path.exists(f'{DATA}/segments.json') else {}

scen = [json.loads(l) for l in open(f'{DATA}/scenarios.jsonl', encoding='utf-8')]
items = [s for s in scen if s['split'] == 'test']
# LIMIT must sample PAIRS, not items.
#
# Two bugs were found here, the second only by simulation:
#
# 1. Taking the first N items returned zero counterfactuals -- they are appended
#    after the base scenarios, so the first 595 test items contain none. A trial
#    run with LIMIT=50 gave NaN for profile_sensitivity and counterfactual
#    correctness, the two metrics the benchmark exists to compute.
#
# 2. Shuffling then slicing fixes that but still fails, because a counterfactual
#    PAIR needs BOTH members present. Random sampling of 100 items yielded only
#    6 usable pairs -- far too few to estimate anything.
#
# So: sample counterfactuals first, always pull in their origin, then fill the
# remainder with base scenarios.
if LIMIT:
    import random
    rng = random.Random(0)
    by_id = {s['scenario_id']: s for s in items}
    cfs   = [s for s in items if s['counterfactual_of']]
    rng.shuffle(cfs)

    chosen, seen = [], set()
    for cf in cfs:
        origin = by_id.get(cf['counterfactual_of'])
        if origin is None or len(chosen) + 2 > LIMIT * 0.6:
            continue
        for s in (origin, cf):
            if s['scenario_id'] not in seen:
                seen.add(s['scenario_id']); chosen.append(s)

    rest = [s for s in items if s['scenario_id'] not in seen]
    rng.shuffle(rest)
    chosen += rest[:max(0, LIMIT - len(chosen))]
    items = sorted(chosen, key=lambda s: s['scenario_id'])

    ids   = {s['scenario_id'] for s in items}
    pairs = sum(1 for s in items
                if s['counterfactual_of'] and s['counterfactual_of'] in ids)
    print(f'LIMIT={LIMIT}: {pairs} complete counterfactual pairs')
    if pairs < 15:
        print('  WARNING: fewer than 15 pairs -- PS/CC will be very noisy.')
        print('  Raise LIMIT to 200+ or drop it entirely for the reported run.')

print(f'{len(items)} test scenarios, {len(clients)} clients')
# The checkpoint key must encode WHICH items were evaluated, not just the model
# and prompt settings. Without this, changing LIMIT (or the sampling rule) while
# keeping the same model reuses the old checkpoint and silently MIXES two
# different samples: a first run of 50 items plus a second of 70 produced a file
# of 145 rows that was then scored as if it were one 100-item run. The metrics
# were computed over a set the paper could not describe.
import hashlib

_fp = hashlib.sha1(
    '|'.join(sorted(s['scenario_id'] for s in items)).encode()
).hexdigest()[:8]
RUN_ID = f'{BASE_ID}__n{len(items)}__{_fp}'
CKPT = f'{WORK}/{RUN_ID}.jsonl'
print('run id    :', RUN_ID)
print('checkpoint:', CKPT)


## 5. Generate — resumable

Rerun this cell after a disconnect; it picks up where it stopped.

In [ ]:
done = {}
if os.path.exists(CKPT):
    for line in open(CKPT, encoding='utf-8'):
        try:
            r = json.loads(line); done[r['scenario_id']] = r
        except json.JSONDecodeError:
            pass   # truncated final line from a hard kill
wanted = {s['scenario_id'] for s in items}
stray = set(done) - wanted
if stray:
    raise SystemExit(
        f'{len(stray)} rows in the checkpoint are not in the current sample. '
        'This should be impossible now that RUN_ID carries a sample fingerprint. '
        f'Delete {CKPT} and rerun.')
print(f'already done: {len(done)} / {len(items)}')

parser = AnswerParser(catalog)

@torch.no_grad()
def generate(system, user):
    msgs = [{'role': 'system', 'content': system},
            {'role': 'user', 'content': user}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    enc = tok(text, return_tensors='pt').to(model.device)
    out = model.generate(**enc, max_new_tokens=200, do_sample=False,
                         pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][enc['input_ids'].shape[1]:], skip_special_tokens=True).strip()

todo = [s for s in items if s['scenario_id'] not in done]
t0, buf = time.time(), []
fh = open(CKPT, 'a', encoding='utf-8')

for i, s in enumerate(todo, 1):
    cid = s['client_id']
    system, user = prompts.build_prompt(
        s, clients[cid].to_dict(), catalog,
        feats=feats.get(cid), segment=segs.get(cid),
        context=CONTEXT, catalogue=CATALOGUE, language=LANGUAGE)
    try:
        text = generate(system, user)
    except Exception as e:
        print(f'  error on {s["scenario_id"]}: {e}')
        text = ''
    p = parser.parse(text)
    fh.write(json.dumps({
        'scenario_id': s['scenario_id'], 'client_id': cid,
        'segment': segs.get(cid), 'answer_text': text,
        'pred_action': p['action'], 'pred_product_id': p['product_id'],
        'asserted_numbers': p['asserted_numbers'],
        'unparseable': p['unparseable'],
    }, ensure_ascii=False) + '\n')
    if i % 25 == 0:
        fh.flush(); os.fsync(fh.fileno())
        el = time.time() - t0
        eta = el / i * (len(todo) - i) / 60
        print(f'  {i}/{len(todo)}  {el/i:.2f}s/item  ETA {eta:.0f} min')
fh.close()
print(f'\ndone in {(time.time()-t0)/60:.1f} min -> {CKPT}')

## 6. Sanity check the raw output

**Read a few answers before trusting any metric.** If the model replies in
Russian to a Kazakh question, or emits an empty string, or repeats the prompt,
the numbers below are measuring that rather than personalisation.

In [ ]:
rows = [json.loads(l) for l in open(CKPT, encoding='utf-8')]
gold = {s['scenario_id']: s for s in scen}

unpars = sum(r['unparseable'] for r in rows) / len(rows)
print(f'unparseable: {unpars:.1%}   (above ~10% means the parser needs work)')
print(f'action distribution: {collections.Counter(r["pred_action"] for r in rows)}')
print(f'distinct products predicted: {len({r["pred_product_id"] for r in rows})}')
print()
for r in rows[:5]:
    g = gold[r['scenario_id']]
    print(f"Q  ({g['intent']}) {g['question_kk']}")
    print(f"gold {g['gold_action']:18s} {g['gold_product_id']}")
    print(f"pred {str(r['pred_action']):18s} {r['pred_product_id']}")
    print(f"     {r['answer_text'][:220]}")
    print('-' * 70)

## 7. Score

In [ ]:
OUT = f'{REPO}/results/preds/{RUN_ID}'
os.makedirs(OUT, exist_ok=True)
!cp "$CKPT" "$OUT/predictions.jsonl"
!python scripts/evaluate.py --data data/v0.5 --predictions "$OUT/predictions.jsonl" --out results/eval/$RUN_ID

## 8. Compare against the reference policies

This is the table that goes in the paper. The question is not whether the
model scores well in absolute terms, but where it sits relative to
`profile_blind`, which achieves 0.708 product-match while ignoring the client
completely.

In [ ]:
m = json.load(open(f'{REPO}/results/eval/{RUN_ID}/metrics.json', encoding='utf-8'))
ref = {
  'oracle (rules)':      dict(pma=1.000, ps=1.000, cc=1.000, cvr=0.000),
  'profile_blind stub':  dict(pma=0.708, ps=0.000, cc=0.000, cvr=0.385),
  'elig_blind stub':     dict(pma=1.000, ps=0.743, cc=0.295, cvr=0.398),
}
print(f"{'system':<24}{'PMA':>8}{'ProfSens':>10}{'CFCorr':>9}{'CVR':>8}")
print('-' * 59)
for k, v in ref.items():
    print(f"{k:<24}{v['pma']:>8.3f}{v['ps']:>10.3f}{v['cc']:>9.3f}{v['cvr']:>8.3f}")
print(f"{MODEL_ID.split('/')[-1]:<24}"
      f"{m['product_match_accuracy']:>8.3f}{m['profile_sensitivity']:>10.3f}"
      f"{m['counterfactual_correctness']:>9.3f}{m['constraint_violation_rate']:>8.3f}")
print()
if m['counterfactual_correctness'] < 0.05:
    print('The model is NOT using the client profile. Same finding as the stub.')
elif m['product_match_accuracy'] < 0.708:
    print('The model scores below a stub that ignores the client entirely.')
else:
    print('Above the blind stub. Check counterfactual correctness for real personalisation.')

## 9. Ablation over context and language

Change `CONTEXT` in cell 3 and rerun cells 3, 5, 7 for each of
`none / persona / profile / full`. Each combination writes its own checkpoint,
so nothing is overwritten.

Two axes matter most for the paper:

* **`none` vs `profile`** — does adding the client profile change anything at
  all? If not, that is the headline result.
* **`kk` vs `ru` vs `mixed`** — the code-switching degradation, run the same
  context level three times.

`persona` reproduces the approach of the earlier paper. Report it honestly
even if it wins.

**Three seeds per configuration.** A single run is a point estimate with no
error bar, which is one of the things the earlier submission was criticised
for. Greedy decoding is deterministic, so vary the *dataset* seed rather than
the sampling seed: rebuild with `--seed 43` and `--seed 44` and rerun.